# AITR Lab 02 — Rerun: translation + Hindi/Odia scoring (protocol v4)

The English test run already succeeded. This notebook does only what's missing.

**Runtime → Change runtime type → T4 GPU.** Then run the cells **one at a time with ▶** (not Run all), so the token step can wait for you.

## 1. GPU + project

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/AITR_upload.zip -d /content/
%cd /content/AITR_Lab02_MAI2026015_MultiSciQA

## 2. Libraries

In [ ]:
!pip install -q 'transformers==4.51.3' 'IndicTransToolkit==1.1.1' sentencepiece sacrebleu pyyaml pytest
import transformers; print('transformers', transformers.__version__)

## 3. Hugging Face token — this cell WAITS for you
Paste your token into the box that appears and press Enter. Nothing is shown while you paste; that's normal.

In [ ]:
from getpass import getpass
from huggingface_hub import login, whoami, hf_hub_download
login(token=getpass('Paste HF token, then Enter: '), add_to_git_credential=False)
print('Logged in as:', whoami()['name'])

## 4. Access check — must print `access OK` twice before continuing

In [ ]:
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError
ok = True
for repo in ['ai4bharat/indictrans2-en-indic-dist-200M', 'ai4bharat/indictrans2-indic-en-dist-200M']:
    try:
        hf_hub_download(repo, 'config.json'); print(repo, '-> access OK')
    except (GatedRepoError, HfHubHTTPError) as e:
        ok = False; print(repo, '-> NO ACCESS:', str(e).splitlines()[0])
if not ok:
    raise SystemExit('STOP: open the model page while logged in as this user and accept the terms. '
                     'If you use a fine-grained token, enable "Read access to contents of all public gated repos you can access".')

## 5. Tests (must end with `passed`)

In [ ]:
!python -m pytest -q 2>&1 | tail -2

## 6. Translate Hindi, then Odia (about 10–20 min each)

In [ ]:
!python scripts/01_translate.py --lang hi --split test 2>&1 | grep -v -i -E 'warn|^\s*$' | tail -30

In [ ]:
!python scripts/01_translate.py --lang or --split test 2>&1 | grep -v -i -E 'warn|^\s*$' | tail -30

## 7. Score NATIVE and TT for Hindi and Odia (errors are shown in full)

In [ ]:
import subprocess, json
for lang in ['hi', 'or']:
    for cond in ['NATIVE', 'TT']:
        r = subprocess.run(['python', 'scripts/02_score.py', '--model', 'Qwen/Qwen2.5-1.5B-Instruct', '--condition', cond, '--lang', lang,
                            '--split', 'test', '--confirm-test'], capture_output=True, text=True)
        if r.returncode != 0:
            print('FAILED', cond, lang); print(r.stderr[-3000:]); raise SystemExit(1)
        m = json.load(open(f'results/preds/Qwen__Qwen2.5-1.5B-Instruct/test_{cond}_{lang}_P0.meta.json'))
        print(cond, lang, '| acc', m['accuracy'], '| macro-F1', m['macro_f1'], '| seconds', m['seconds'])

## 8. Save to Drive

In [ ]:
!zip -q -r /content/drive/MyDrive/results_part2.zip data/translations results/translation results/preds
print('Saved to My Drive: results_part2.zip')

## 9. Done — download **results_part2.zip** into the `Desktop/KOUSAL_ATR` folder (or Downloads) and tell Claude.